In [2]:
using Test
using Enzyme
using LinearAlgebra
using Random

using NeuroDSL


function test_enzyme_integration_robust()
    # Vos buffers habituels
    x = randn(Float32, 10, 10)
    dy = randn(Float32, 10, 10)
    dx = zeros(Float32, 10, 10)
    
    # 1. Votre fonction "Black Box"
    f_relu(x) = max.(x, 0f0)
    
    # 2. Le "Wrapper Scalaire" (Le secret des moteurs d'AD)
    # On définit une fonction qui calcule le produit scalaire (dy * f(x))
    # Enzyme va différentier cette fonction scalaire par rapport à x
    function scalar_wrapper(x)
        y = f_relu(x)
        return dot(dy, y)
    end
    
    # 3. Appel de Enzyme sur le wrapper
    # Enzyme va remplir dx automatiquement avec le gradient de `dot(dy, f(x))`
    Enzyme.autodiff(Enzyme.Reverse, scalar_wrapper, Enzyme.Duplicated(x, dx))
    
    # Vérification
    manuelle_grad = dy .* (x .> 0f0)
    @test isapprox(manuelle_grad, dx, atol=1e-6)
    println("✅ ReLU : Match manuel/enzyme avec scalar_wrapper")
    
    return dx
end

test_enzyme_integration_robust()

✅ ReLU : Match manuel/enzyme avec scalar_wrapper


10×10 Matrix{Float32}:
  0.0448472   0.0       -0.370729  …   1.02045    -0.391508  -0.445197
  0.734872    0.0        0.827186      0.381029    1.2229     0.0
  0.0         0.0        0.926871     -0.0844609   0.0        0.609207
 -1.04264     0.0        0.0           0.0         0.0        0.0
  0.0        -0.87252   -2.31053      -0.574355    0.0        0.0
  0.0104246  -0.325012   0.0       …  -1.57773    -0.594796   0.747915
  0.0        -0.412991   0.0           0.0         0.564223   0.0
  0.759717    0.0        0.0           0.0         0.0        0.0
  0.618369    0.0       -0.528895     -0.834344    0.847888   0.0
  0.0         0.444117   0.0           0.0         0.0        0.0

In [4]:
using BenchmarkTools
using Enzyme
using LinearAlgebra
using Random
using NeuroDSL
# Préparation des données (Float32 pour correspondre au Backend de NeuroDSL)
Random.seed!(42)
M, K, N = 128, 64, 128
X = randn(Float32, M, K)
W = randn(Float32, K, N)
dy = randn(Float32, M, N)

# 1. IMPLÉMENTATION MANUELLE (Inspirée de vos GRAD_RULES dans backward.jl)[cite: 3]
function classic_grad(X, W, dy)
    # Forward pass (nécessaire pour le masque)
    out = max.(X * W, 0.0f0)
    
    # ReLU backward: gradient est nul là où sortie <= 0
    dz = dy .* (out .> 0f0)
    
    # MatMul Backward
    dX = dz * W'
    dW = X' * dz
    return dX, dW
end

# 2. IMPLÉMENTATION ENZYME (via Scalar Wrapper)
function enzyme_grad(X, W, dy)
    dX = zeros(Float32, size(X))
    dW = zeros(Float32, size(W))
    
    # Forward + Scalar Wrapper
    function scalar_wrapper(X, W)
        y = max.(X * W, 0.0f0)
        return dot(dy, y)
    end
    
    Enzyme.autodiff(Enzyme.Reverse, scalar_wrapper, 
                    Enzyme.Duplicated(X, dX), 
                    Enzyme.Duplicated(W, dW))
    return dX, dW
end

# --- LANCEMENT DU BENCHMARK ---
println("--- Lancement du Benchmark (Forward + Backward fusionné) ---")

print("Classic (Manuel) : ")
bench_classic = @btime classic_grad($X, $W, $dy)

print("Enzyme (Compilé) : ")
bench_enzyme = @btime enzyme_grad($X, $W, $dy)

# Vérification de l'équivalence des résultats
dX_c, dW_c = classic_grad(X, W, dy)
dX_e, dW_e = enzyme_grad(X, W, dy)

@assert isapprox(dX_c, dX_e, atol=1e-5)
@assert isapprox(dW_c, dW_e, atol=1e-5)
println("\n✅ Les résultats sont identiques (tolérance 1e-5).")

--- Lancement du Benchmark (Forward + Backward fusionné) ---
Classic (Manuel) :   515.000 μs (10 allocations: 256.23 KiB)
Enzyme (Compilé) :   697.500 μs (12 allocations: 320.28 KiB)

✅ Les résultats sont identiques (tolérance 1e-5).


In [7]:
using BenchmarkTools
using Enzyme
using LinearAlgebra
using Random

# --- 1. Simulation du BufferPool (référence NeuroDSL v4/compiler.jl) ---
# Dans votre vrai code, utilisez le BufferPool réel défini dans compiler.jl
struct BufferPool
    dev::Any 
end
acquire!(p::BufferPool, shape) = zeros(Float32, shape...) 
release!(p::BufferPool, buf) = nothing

pool = BufferPool(nothing) 

# --- 2. Données ---
Random.seed!(42)
M, K, N = 128, 64, 128
X = randn(Float32, M, K)
W = randn(Float32, K, N)
dy = randn(Float32, M, N)

# --- 3. Benchmark : Fonctions avec noms synchronisés ---

# Version classique
function classic_grad(X, W, dy)
    out = max.(X * W, 0.0f0)
    dz = dy .* (out .> 0f0)
    dX = dz * W'
    dW = X' * dz
    return dX, dW
end

# Version Enzyme optimisée (Nom synchronisé avec l'appel ci-dessous)
function enzyme_grad_optimized(pool, X::Matrix{Float32}, W::Matrix{Float32}, dy::Matrix{Float32})
    dX = acquire!(pool, size(X))
    dW = acquire!(pool, size(W))
    
    function scalar_wrapper(X::Matrix{Float32}, W::Matrix{Float32})
        y = max.(X * W, 0.0f0)
        return dot(dy, y)
    end
    
    Enzyme.autodiff(Enzyme.Reverse, scalar_wrapper, 
                    Enzyme.Duplicated(X, dX), 
                    Enzyme.Duplicated(W, dW))
    return dX, dW
end

# --- 4. Exécution ---
println("Warming up Enzyme...")
for i in 1:100; enzyme_grad_optimized(pool, X, W, dy); end

println("--- Benchmark Optimisé ---")
# On utilise le nom exact : enzyme_grad_optimized
print("Enzyme (Optimisé) : ")
@btime enzyme_grad_optimized($pool, $X, $W, $dy)

print("Classic (Manuel) : ")
@btime classic_grad($X, $W, $dy)

Warming up Enzyme...
--- Benchmark Optimisé ---
Enzyme (Optimisé) :   704.800 μs (12 allocations: 320.28 KiB)
Classic (Manuel) :   511.900 μs (10 allocations: 256.23 KiB)


(Float32[11.384794 8.6156025 … -0.71443135 -3.7601044; 0.9353655 10.948009 … 4.668751 4.55766; … ; -7.617338 4.8289514 … -2.3520496 1.006053; 3.4625013 -1.4319973 … -2.910364 -3.5322526], Float32[-2.4673214 -2.648581 … 1.0849252 4.6916623; -4.228261 6.131429 … 3.3637798 1.8147508; … ; -14.968233 4.64208 … 10.851605 4.128917; -31.805206 2.8234692 … 1.6858959 -6.0293016])

In [11]:
using BenchmarkTools
using Enzyme
using LinearAlgebra
using Random
using Test

# 1. Setup du BufferPool
if !isdefined(Main, :BufferPool)
    struct BufferPool; dev::Any; end
    acquire!(p::BufferPool, shape) = zeros(Float32, shape...) 
    release!(p::BufferPool, buf) = nothing
end
pool = BufferPool(nothing)

# 2. Données
Random.seed!(42)
N = 1024
gate = randn(Float32, N)
up   = randn(Float32, N)
dy   = randn(Float32, N)

# 3. Implémentation Manuelle (Logique de kernels.jl)[cite: 10]
function swiglu_manual(gate, up, dy)
    n = length(dy)
    dgate = zeros(Float32, n)
    dup = zeros(Float32, n)
    for i in 1:n
        g = gate[i]
        sig = 1.0f0 / (1.0f0 + exp(-g))
        dup[i] = dy[i] * g * sig
        dgate[i] = dy[i] * up[i] * sig * (1.0f0 + g * (1.0f0 - sig))
    end
    return dgate, dup
end

# 4. Implémentation Enzyme (Optimisée : Loop-Fused)
function swiglu_enzyme_fused(pool, gate::Vector{Float32}, up::Vector{Float32}, dy::Vector{Float32})
    dgate = acquire!(pool, size(gate))
    dup   = acquire!(pool, size(up))
    
    # Wrapper avec boucle explicite pour éviter les allocations de broadcasting
    function scalar_wrapper_fused(gate::Vector{Float32}, up::Vector{Float32})
        n = length(gate)
        res = 0.0f0
        for i in 1:n
            g = gate[i]
            sig = 1.0f0 / (1.0f0 + exp(-g))
            # On accumule le résultat du produit scalaire (dy * y)
            res += dy[i] * (g * sig * up[i])
        end
        return res
    end
    
    # Enzyme différentie à travers la boucle et l'accumulation
    Enzyme.autodiff(Enzyme.Reverse, scalar_wrapper_fused, 
                    Enzyme.Duplicated(gate, dgate), 
                    Enzyme.Duplicated(up, dup))
    return dgate, dup
end

# 5. Exécution
println("Warming up Enzyme...")
for i in 1:100; swiglu_enzyme_fused(pool, gate, up, dy); end

println("--- Benchmark SwiGLU (Loop-Fused) ---")
print("Manuel (kernels.jl) : ")
@btime swiglu_manual($gate, $up, $dy)

print("Enzyme (Optimisé) : ")
@btime swiglu_enzyme_fused($pool, $gate, $up, $dy)

# Vérification
dgate_m, dup_m = swiglu_manual(gate, up, dy)
dgate_e, dup_e = swiglu_enzyme_fused(pool, gate, up, dy)

@test isapprox(dgate_m, dgate_e, atol=1e-5)
println("\n✅ Résultats identiques.")

Warming up Enzyme...
--- Benchmark SwiGLU (Loop-Fused) ---
Manuel (kernels.jl) :   5.433 μs (2 allocations: 8.25 KiB)
Enzyme (Optimisé) :   18.500 μs (2 allocations: 8.25 KiB)

✅ Résultats identiques.


In [17]:
using Test
using Enzyme
using LinearAlgebra
using NeuroDSL
using Logging # Pour gérer proprement les warnings

# --- 1. SETUP DE BASE ---
if !isdefined(Main, :BufferPool)
    struct BufferPool; dev::Any; end
    acquire!(p::BufferPool, shape) = zeros(Float32, shape...) 
    release!(p::BufferPool, buf) = nothing
end

if !isdefined(Main, :pool)
    const pool = BufferPool(nothing)
end

# --- 2. BRIDGE ENZYME (Backward) ---
function register_enzyme_rule!(op_name::Symbol, f::Function)
    GRAD_RULES[op_name] = (dev, dy, ctx, inputs) -> begin
        d_inputs = [similar(x) for x in inputs]
        function scalar_wrapper(args...)
            y = f(args...)
            return dot(dy, y)
        end
        Enzyme.autodiff(Enzyme.Reverse, scalar_wrapper, 
                        [Enzyme.Duplicated(in_val, d_in) for (in_val, d_in) in zip(inputs, d_inputs)]...)
        return tuple(d_inputs...)
    end
end

# --- 3. OPÉRATEURS (Forward) ---
function relu_fwd_op(dev, output_buffer, inputs, attrs, out_sym, out_node, ctx_store)
    output_buffer .= max.(inputs[1], 0f0)
    return output_buffer
end

# --- 4. EXÉCUTION DU TEST ---
function run_full_integration_test_clean()
    # Enregistrement des composants
    register_op!(:relu_enzyme, relu_fwd_op)
    register_enzyme_rule!(:relu_enzyme, x -> max.(x, 0f0))

    # Initialisation du graphe
    g = NeuroGraph(device = NeuroDSL.Backend.CPUDevice())
    activate!(g, :test_env)

    # Setup
    x_val = randn(Float32, 5, 5)
    set!(g, :x, x_val, is_param=true)
    
    # Ajout de la règle
    addrule!(g, GraphRule(:y, [:x], :relu_enzyme; namespace=:test_env))

    # Exécution avec suppression temporaire des warnings de Shape Inference
    with_logger(SimpleLogger(stderr, Logging.Error)) do
        demand!(g, :y; namespace=:test_env)
        backward_graph!(g, :y; namespace=:test_env)
    end

    # Validation
    x_node = node(g, :x; namespace=:test_env)
    @test x_node.gradient !== nothing
    
    expected_grad = (x_val .> 0f0)
    @test isapprox(x_node.gradient, expected_grad, atol=1e-5)
    
    println("✅ Intégration Enzyme validée et console nettoyée !")
end

run_full_integration_test_clean()

✅ Op :relu_enzyme registered
✅ Intégration Enzyme validée et console nettoyée !


In [21]:
using Test
using Enzyme
using LinearAlgebra
using NeuroDSL
using Logging

# --- 1. SETUP DE BASE (Idempotent) ---
if !isdefined(Main, :BufferPool)
    struct BufferPool; cache::Dict{Dims, Vector{Array{Float32}}}; end
    
    function acquire!(p::BufferPool, shape)
        if haskey(p.cache, shape) && !isempty(p.cache[shape])
            return pop!(p.cache[shape])
        end
        return zeros(Float32, shape)
    end
end

if !isdefined(Main, :pool)
    const pool = BufferPool(Dict{Dims, Vector{Array{Float32}}}())
end

# --- 2. FONCTIONS NOMMÉES (Élimination des allocations de closures) ---

# Règle de Gradient (Backward) - FONCTION NOMMÉE
function relu_grad_rule(dev, dy, ctx, inputs)
    # On acquiert les buffers via le pool pour éviter 'similar()'
    d_inputs = [acquire!(pool, size(x)) for x in inputs]
    
    # Wrapper simple sans capture de variables complexes
    function scalar_wrapper(args...)
        y = max.(args[1], 0f0)
        return dot(dy, y)
    end
    
    # Calcul du gradient
    Enzyme.autodiff(Enzyme.Reverse, scalar_wrapper, 
                    [Enzyme.Duplicated(in_val, d_in) for (in_val, d_in) in zip(inputs, d_inputs)]...)
    
    return tuple(d_inputs...)
end

# Opération Forward
function relu_fwd_op(dev, output_buffer, inputs, attrs, out_sym, out_node, ctx_store)
    output_buffer .= max.(inputs[1], 0f0)
    return nothing 
end

# --- 3. EXÉCUTION DU TEST ---
function run_full_integration_test_clean()
    # Enregistrement
    register_op!(:relu_enzyme, relu_fwd_op)
    GRAD_RULES[:relu_enzyme] = relu_grad_rule

    # Initialisation
    g = NeuroGraph(device = NeuroDSL.Backend.CPUDevice())
    activate!(g, :test_env)

    # Setup
    x_val = randn(Float32, 5, 5)
    set!(g, :x, x_val, is_param=true)
    addrule!(g, GraphRule(:y, [:x], :relu_enzyme; namespace=:test_env))

    # Exécution avec suppression des warnings
    # On filtre les warnings de 'Shape inference' pour avoir une console propre
    with_logger(SimpleLogger(stderr, Logging.Error)) do
        demand!(g, :y; namespace=:test_env)
        backward_graph!(g, :y; namespace=:test_env)
    end

    # Validation
    x_node = node(g, :x; namespace=:test_env)
    @test x_node.gradient !== nothing
    expected_grad = (x_val .> 0f0)
    @test isapprox(x_node.gradient, expected_grad, atol=1e-5)
    
    println("✅ Intégration Enzyme optimisée (Zero-Alloc approach) validée !")
end

run_full_integration_test_clean()

✅ Op :relu_enzyme registered
✅ Intégration Enzyme optimisée (Zero-Alloc approach) validée !


In [25]:
using Test
using Enzyme
using LinearAlgebra
using NeuroDSL
using BenchmarkTools
using Logging

# --- 1. INFRASTRUCTURE (Idempotente) ---
if !isdefined(Main, :BufferPool)
    struct BufferPool; cache::Dict{Dims, Vector{Array{Float32}}}; end
    
    function acquire!(p::BufferPool, shape)
        if haskey(p.cache, shape) && !isempty(p.cache[shape])
            return pop!(p.cache[shape])
        end
        return zeros(Float32, shape)
    end
end

if !isdefined(Main, :pool)
    const pool = BufferPool(Dict{Dims, Vector{Array{Float32}}}())
end

# --- 2. FONCTIONS DE CALCUL (Top-level pour Enzyme) ---

# La fonction de calcul : f(x)
function f_relu(x)
    return max.(x, 0f0)
end

# La fonction de perte pour Enzyme : L = dot(dy, f(x))
# On passe 'dy' en Const car on ne veut pas le différencier
function relu_loss_function(dy, x)
    y = f_relu(x)
    return dot(dy, y)
end

# --- 3. RÈGLE DE GRADIENT ---
function relu_grad_rule(dev, dy, ctx, inputs)
    # Récupération des buffers
    d_inputs = [acquire!(pool, size(x)) for x in inputs]
    
    # Appel Enzyme direct sur la fonction top-level
    # On passe dy en Const (on ne veut pas son gradient)
    # On passe l'input en Duplicated (pour remplir d_inputs)
    Enzyme.autodiff(Enzyme.Reverse, 
                    relu_loss_function, 
                    Enzyme.Const(dy), 
                    Enzyme.Duplicated(inputs[1], d_inputs[1]))
    
    return tuple(d_inputs...)
end

# --- 4. OPÉRATEURS & CONFIGURATION ---
function relu_fwd_op(dev, output_buffer, inputs, attrs, out_sym, out_node, ctx_store)
    output_buffer .= max.(inputs[1], 0f0)
    return nothing
end

function setup_environment()
    register_op!(:relu_enzyme, relu_fwd_op)
    GRAD_RULES[:relu_enzyme] = relu_grad_rule
    println("✅ Environnement configuré.")
end

# --- 5. BENCHMARK ROBUSTE ---
function run_full_benchmark(N=512)
    # Création du graphe
    g = NeuroGraph(device = NeuroDSL.Backend.CPUDevice())
    activate!(g, :bench)
    
    x_val = randn(Float32, N, N)
    set!(g, :x, x_val, is_param=true)
    addrule!(g, GraphRule(:y, [:x], :relu_enzyme; namespace=:bench))
    
    # WARMUP
    for i in 1:20
        demand!(g, :y; namespace=:bench)
        backward_graph!(g, :y; namespace=:bench)
    end
    
    # BENCHMARK
    println("--- Benchmark (N=$N) ---")
    trial = @benchmark begin
        demand!($g, :y; namespace=:bench)
        backward_graph!($g, :y; namespace=:bench)
    end samples=100
    
    display(trial)
    return trial
end

# --- 6. EXÉCUTION ---
setup_environment()
run_full_benchmark(512)

✅ Op :relu_enzyme registered
✅ Environnement configuré.


┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Ne

--- Benchmark (N=512) ---


┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Ne

BenchmarkTools.Trial: 100 samples with 1 evaluation per sample.
 Range (min … max):  4.515 ms …   9.137 ms  ┊ GC (min … max): 0.00% … 38.92%
 Time  (median):     4.952 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   5.227 ms ± 968.540 μs  ┊ GC (mean ± σ):  4.83% ± 10.46%

    ▁▁█ ▇▂▁                                                    
  ▄▇████████▄▆▅▄▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▃▁▄▃▁▁▁▃▄ ▃
  4.52 ms         Histogram: frequency by time        8.42 ms <

 Memory estimate: 5.01 MiB, allocs estimate: 208.

BenchmarkTools.Trial: 100 samples with 1 evaluation per sample.
 Range (min … max):  4.515 ms …   9.137 ms  ┊ GC (min … max): 0.00% … 38.92%
 Time  (median):     4.952 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   5.227 ms ± 968.540 μs  ┊ GC (mean ± σ):  4.83% ± 10.46%

    ▁▁█ ▇▂▁                                                    
  ▄▇████████▄▆▅▄▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▃▁▄▃▁▁▁▃▄ ▃
  4.52 ms         Histogram: frequency by time        8.42 ms <

 Memory estimate: 5.01 MiB, allocs estimate: 208.

In [28]:
using Test
using Enzyme
using LinearAlgebra
using NeuroDSL
using Profile
using Logging

# --- 1. SETUP DE BASE (Idempotent) ---
if !isdefined(Main, :BufferPool)
    struct BufferPool; cache::Dict{Dims, Vector{Array{Float32}}}; end
    function acquire!(p::BufferPool, shape)
        if haskey(p.cache, shape) && !isempty(p.cache[shape])
            return pop!(p.cache[shape])
        end
        return zeros(Float32, shape)
    end
end
if !isdefined(Main, :pool)
    const pool = BufferPool(Dict{Dims, Vector{Array{Float32}}}())
end

# --- 2. LOGIQUE (Fonctions nommées pour zéro-alloc) ---
function f_relu(x) return max.(x, 0f0) end
function relu_loss_function(dy, x) return dot(dy, f_relu(x)) end

function relu_grad_rule(dev, dy, ctx, inputs)
    d_inputs = [acquire!(pool, size(x)) for x in inputs]
    Enzyme.autodiff(Enzyme.Reverse, relu_loss_function, Enzyme.Const(dy), Enzyme.Duplicated(inputs[1], d_inputs[1]))
    return tuple(d_inputs...)
end

function relu_fwd_op(dev, output_buffer, inputs, attrs, out_sym, out_node, ctx_store)
    output_buffer .= max.(inputs[1], 0f0)
    return nothing
end

# --- 3. PROFILAGE DU SYSTÈME ---
function run_profiling_diagnostic()
    # Initialisation locale de 'g' pour éviter l'UndefVarError
    g = NeuroGraph(device = NeuroDSL.Backend.CPUDevice())
    activate!(g, :bench)
    
    # Setup
    x_val = randn(Float32, 512, 512)
    set!(g, :x, x_val, is_param=true)
    
    # Enregistrement
    register_op!(:relu_enzyme, relu_fwd_op)
    GRAD_RULES[:relu_enzyme] = relu_grad_rule
    addrule!(g, GraphRule(:y, [:x], :relu_enzyme; namespace=:bench))
    
    # Warmup
    for i in 1:10
        demand!(g, :y; namespace=:bench)
        backward_graph!(g, :y; namespace=:bench)
    end

    # PROFILAGE
    println("--- Lancement du profilage (100 itérations) ---")
    Profile.clear()
    @profile for i in 1:100
        backward_graph!(g, :y; namespace=:bench)
    end
    
    # Affichage des résultats
    # format=:flat trie par fonction la plus lourde
    println("--- Résultat du profilage (Fonctions les plus lourdes) ---")
    Profile.print(format=:flat, C=false)
end

# Exécution
run_profiling_diagnostic()

✅ Op :relu_enzyme registered


┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Ne

--- Lancement du profilage (100 itérations) ---


┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Ne

--- Résultat du profilage (Fonctions les plus lourdes) ---


┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Nevermind\Desktop\NeuroDSL\src\dispatch.jl:94
┌ Warning: Shape inference non implémentée pour :relu_enzyme, utilisation de la forme du premier argument
└ @ NeuroDSL C:\Users\Ne

 Count  Overhead File                    Line Function
 =====  ======== ====                    ==== ========
     7         0 In[28]                     ? diffejulia_relu_loss_function_622…
   182         0 In[28]                    23 f_relu
    62         0 In[28]                    62 macro expansion
    14         0 In[28]                    27 relu_grad_rule(dev::NeuroDSL.Back…
     7         0 In[28]                    28 relu_grad_rule(dev::NeuroDSL.Back…
     7         0 In[28]                    24 relu_loss_function
    62         0 In[28]                    61 run_profiling_diagnostic()
    14         0 In[7]                     11 acquire!
    16         0 @Base\abstractarray.jl   876 similar
    16         0 @Base\abstractarray.jl   877 similar
     1         0 @Base\array.jl           723 _array_for
     1         0 @Base\array.jl           726 _array_for
    17         0 @Base\array.jl           376 _copyto_impl!(dest::Matrix{Float3…
    14         0 @Base\array.jl     